<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/Taller_Control_1.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab">
</a>

# Taller domiciliario de control 1 — S1 a S6
## Construir un histórico SECOP y convertirlo en una solución NoSQL

**Parejas · 100 puntos · Git/GitHub opcional**

En este taller **no van a repetir los ejercicios de clase**.

En clase trabajaron con una muestra de 1.000 procesos y practicaron MongoDB/Atlas, Cassandra y Neo4j. Aquí recibirán **seis archivos de 1.000 filas** y deberán construir un histórico nuevo de **6.000 registros**, convertirlo a documentos JSON, cargarlo **realmente en MongoDB Atlas** y resolver preguntas nuevas.

Después, una salida de Atlas alimentará el diseño Cassandra y el análisis relacional.

### Flujo del taller

`6 CSV → histórico 6.000 → JSON → Atlas → consultas → bandeja → Cassandra → Neo4j → informe`

> La guía completa también está disponible en `Talleres/Taller_Control_1.md`, pero **cada etapa de este notebook contiene las instrucciones necesarias para resolverla**.


In [ ]:
from pathlib import Path
import json, re, hashlib, urllib.request, subprocess, sys
import pandas as pd
import numpy as np

INTEGRANTE_1 = input("Integrante 1 - nombre: ").strip()
CODIGO_1 = input("Integrante 1 - código: ").strip()
INTEGRANTE_2 = input("Integrante 2 - nombre: ").strip()
CODIGO_2 = input("Integrante 2 - código: ").strip()

if not all([INTEGRANTE_1, CODIGO_1, INTEGRANTE_2, CODIGO_2]):
    raise ValueError("Completen nombres y códigos de ambos integrantes.")

def slug(texto):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(texto)).strip("_").lower()

PAREJA_ID = "_".join(sorted([slug(CODIGO_1), slug(CODIGO_2)]))
OUT = Path("entrega_tc1")
OUT.mkdir(exist_ok=True)

DATA_COMMIT = "c7031e3a58daa22d4ceff2d2f01d66aa967ba9e3"
RAW = f"https://raw.githubusercontent.com/jazaineam1/BigData2026/{DATA_COMMIT}"

CHUNKS = [
    "prueba_chunk_0000000.csv",
    "prueba_chunk_0001000.csv",
    "prueba_chunk_0002000.csv",
    "prueba_chunk_0003000.csv",
    "prueba_chunk_0004000.csv",
    "prueba_chunk_0005000.csv",
]
URLS_SECOP = {
    nombre: f"{RAW}/Cuadernos/datos/secop_chunks/{nombre}"
    for nombre in CHUNKS
}

print("Pareja:", PAREJA_ID)
print("Archivos asignados:", len(URLS_SECOP))
print("Carpeta de entrega:", OUT.resolve())


---
# ETAPA 1 — Construyan un histórico nuevo de SECOP
**20 puntos**

## Qué deben demostrar

En clase cargaron una muestra de 1.000 filas. Ahora deben demostrar que saben **integrar varias fuentes con la misma estructura**, limpiar tipos y producir documentos listos para MongoDB.

Van a usar estos seis archivos:

- `prueba_chunk_0000000.csv`
- `prueba_chunk_0001000.csv`
- `prueba_chunk_0002000.csv`
- `prueba_chunk_0003000.csv`
- `prueba_chunk_0004000.csv`
- `prueba_chunk_0005000.csv`

Cada archivo tiene 1.000 filas. **No los unan manualmente en Excel.**

## Paso 1.1 — Cargar y concatenar

Escriban código que:

1. recorra `URLS_SECOP`;
2. lea cada CSV con `pd.read_csv(..., low_memory=False)`;
3. agregue `archivo_origen` con el nombre del CSV;
4. guarde cada DataFrame en `fragmentos`;
5. concatene todo en `secop_historico`.

### Debe verse en la salida

Para cada archivo impriman: nombre, filas y columnas.

Al final impriman, calculado por código:

- cantidad de archivos;
- filas integradas;
- procesos únicos (`id_del_proceso`);
- departamentos distintos.


In [ ]:
# ============================================================
# ETAPA 1.1 — SU CÓDIGO
# ============================================================

fragmentos = []
secop_historico = None

# Escriban aquí la carga de los seis CSV.
# No escriban "6000" manualmente en ninguna variable de control.


## Paso 1.2 — Crear el esquema documental

Desde `secop_historico` creen un DataFrame llamado `historico`.

Debe contener **exactamente estos 16 campos**:

| Campo final | Origen |
|---|---|
| `id_proceso` | `id_del_proceso` |
| `entidad` | `entidad` |
| `nit_entidad` | `nit_entidad` |
| `departamento` | `departamento_entidad` |
| `ciudad` | `ciudad_entidad` |
| `fecha_publicacion` | `fecha_de_publicacion` |
| `anio` | derivado de `fecha_publicacion` |
| `precio_base` | `precio_base` |
| `modalidad` | `modalidad_de_contratacion` |
| `respuestas` | `respuestas_al_procedimiento` |
| `estado` | `estado_del_procedimiento` |
| `adjudicado` | `adjudicado` |
| `proveedor` | `nombre_del_proveedor` |
| `nit_proveedor` | `nit_del_proveedor_adjudicado` |
| `url` | `urlproceso` |
| `archivo_origen` | creado por ustedes |

### Limpieza obligatoria

- `fecha_publicacion`: `pd.to_datetime(..., errors="coerce")`;
- `anio`: año derivado de la fecha;
- `precio_base`: `pd.to_numeric(..., errors="coerce")`;
- `respuestas`: `pd.to_numeric(..., errors="coerce")`;
- `adjudicado`: booleano `True/False`;
- faltantes: deben terminar como `null` en JSON, nunca como texto `"nan"`.


In [ ]:
# ============================================================
# ETAPA 1.2 — SU CÓDIGO
# ============================================================

historico = None

# 1. Seleccionen y renombren las columnas.
# 2. Conviertan fechas y campos numéricos.
# 3. Conviertan adjudicado a booleano.
# 4. Muestren historico.head() y historico.dtypes.


## Paso 1.3 — Generar el JSON que después cargarán en Atlas

Creen `documentos`, una lista de diccionarios.

El archivo `01_secop_historico_6000.json` debe ser un **arreglo JSON** con documentos de esta forma:

```json
[
  {
    "id_proceso": "CO1.REQ....",
    "entidad": "...",
    "nit_entidad": "...",
    "departamento": "...",
    "ciudad": "...",
    "fecha_publicacion": "2025-07-23T00:00:00.000",
    "anio": 2025,
    "precio_base": 250000000.0,
    "modalidad": "...",
    "respuestas": 0,
    "estado": "...",
    "adjudicado": true,
    "proveedor": "...",
    "nit_proveedor": "...",
    "url": "https://...",
    "archivo_origen": "prueba_chunk_0000000.csv"
  }
]
```

**El ejemplo muestra la estructura, no una respuesta.**

Pueden convertir de DataFrame a documentos con:

```python
documentos = json.loads(
    historico.to_json(
        orient="records",
        force_ascii=False,
        date_format="iso"
    )
)
```

Después creen `control_ingesta` con valores calculados:

```python
control_ingesta = {
    "archivos_cargados": ...,
    "filas_integradas": ...,
    "procesos_unicos": ...,
    "departamentos": ...,
    "anio_min": ...,
    "anio_max": ...,
    "documentos_json": ...
}
```

Guarden:

- `entrega_tc1/01_secop_historico_6000.json`
- `entrega_tc1/01_control_ingesta.json`


In [ ]:
# ============================================================
# ETAPA 1.3 — SU CÓDIGO
# ============================================================

documentos = None
control_ingesta = {}

# Generen documentos y los dos archivos JSON.
# Al final impriman control_ingesta.


---
# ETAPA 2 — Carguen el histórico en MongoDB Atlas y hagan consultas nuevas
**30 puntos**

## Qué deben demostrar

En esta etapa **Atlas sí es obligatorio**.

Deben demostrar que pueden:

1. conectarse a su propio clúster;
2. crear una colección;
3. cargar los documentos que ustedes prepararon;
4. verificar la carga;
5. hacer un `count_documents`;
6. hacer un `find` con filtro, proyección, orden y límite;
7. construir un `aggregate`.

No usen `mongomock` para esta etapa.

## Paso 2.1 — Conectarse de forma segura

Usen la misma técnica practicada en clase:

```python
from getpass import getpass
from urllib.parse import quote_plus
from pymongo import MongoClient
```

Pidan con `input()` la plantilla SRV que entrega Atlas. Debe parecerse a:

```text
mongodb+srv://<db_username>:<db_password>@cluster....
```

Pidan usuario con `input()` y contraseña con `getpass()`.

Reemplacen `<db_username>` y `<db_password>` en la plantilla.

**No escriban la contraseña directamente en el notebook.**

Al final deben existir:

- `client`
- `atlas_ping`
- `atlas_server_version`

y `client.admin.command("ping")` debe responder correctamente.


In [ ]:
# ============================================================
# ETAPA 2.1 — CONEXIÓN REAL A ATLAS
# ============================================================

# Si hace falta:
# !pip -q install pymongo[srv]

client = None
atlas_ping = False
atlas_server_version = None

# Conéctense a Atlas usando input() + getpass().
# Hagan ping y recuperen la versión del servidor.


## Paso 2.2 — Crear su colección y cargar los 6.000 documentos

Usen:

```text
base de datos: tc1_bigdata
colección: secop_historico_<PAREJA_ID>
```

Definan:

```python
db = client["tc1_bigdata"]
NOMBRE_COLECCION = f"secop_historico_{PAREJA_ID}"
coleccion = db[NOMBRE_COLECCION]
```

Para poder ejecutar el notebook varias veces sin duplicar filas:

```python
coleccion.delete_many({})
```

Después carguen **una copia** de los documentos:

```python
coleccion.insert_many([dict(d) for d in documentos])
```

No usen `insert_many(documentos)` directamente porque PyMongo puede agregar `_id` a sus diccionarios originales.

Comprueben:

```python
documentos_atlas = coleccion.count_documents({})
```

Impriman el nombre de la base, colección y cantidad de documentos.


In [ ]:
# ============================================================
# ETAPA 2.2 — CARGA EN ATLAS
# ============================================================

db = None
NOMBRE_COLECCION = f"secop_historico_{PAREJA_ID}"
coleccion = None
documentos_atlas = None

# Creen db y coleccion.
# Vacíen SOLO la colección de la pareja.
# Inserten los documentos.
# Calculen documentos_atlas.


## Paso 2.3 — Consulta A: `count_documents`

Construyan `filtro_a` para contar documentos que cumplan simultáneamente:

- `anio >= 2024`;
- `modalidad` contiene `directa`, ignorando mayúsculas/minúsculas;
- `respuestas == 0`;
- `precio_base > 0`.

Ejecuten:

```python
resultado_a = coleccion.count_documents(filtro_a)
```

El resultado debe salir de Atlas. No se les entrega el número esperado.


In [ ]:
# ============================================================
# ETAPA 2.3 — CONSULTA A
# ============================================================

filtro_a = {}
resultado_a = None

# Construyan el filtro y ejecuten count_documents().
print("Resultado A:", resultado_a)


## Paso 2.4 — Consulta B: `find` + proyección + sort + limit

Busquen documentos con:

- `anio == 2025`;
- `departamento == "Antioquia"`;
- `precio_base > 0`.

La proyección debe mostrar solo:

- `id_proceso`
- `entidad`
- `precio_base`
- `modalidad`

y ocultar `_id`.

Ordenen por:

1. `precio_base DESC`;
2. `id_proceso ASC`.

Limiten a 10.

Deben crear:

- `filtro_b`
- `proyeccion_b`
- `resultado_b` como una lista de documentos.


In [ ]:
# ============================================================
# ETAPA 2.4 — CONSULTA B
# ============================================================

filtro_b = {}
proyeccion_b = {}
resultado_b = []

# Ejecuten find + projection + sort + limit.
resultado_b


## Paso 2.5 — Consulta C: aggregation pipeline

Construyan `pipeline_c`:

### 1. `$match`
- `anio >= 2024`
- `precio_base > 0`

### 2. `$group`
Agrupar por `departamento` y calcular:

- `procesos`: cantidad;
- `valor_total`: suma de `precio_base`;
- `valor_promedio`: promedio de `precio_base`.

### 3. `$sort`
- `procesos DESC`
- `_id ASC`

### 4. `$limit`
- 8

Ejecuten:

```python
resultado_c = list(coleccion.aggregate(pipeline_c))
```


In [ ]:
# ============================================================
# ETAPA 2.5 — CONSULTA C
# ============================================================

pipeline_c = []
resultado_c = []

# Construyan y ejecuten el pipeline.
resultado_c


## Paso 2.6 — Guardar la evidencia de Atlas

Construyan exactamente:

```python
atlas_resultados = {
    "carga": {
        "base_datos": "tc1_bigdata",
        "coleccion": NOMBRE_COLECCION,
        "documentos": documentos_atlas,
        "server_version": atlas_server_version
    },
    "consulta_a": {
        "filtro": filtro_a,
        "resultado": resultado_a
    },
    "consulta_b": {
        "filtro": filtro_b,
        "proyeccion": proyeccion_b,
        "resultado": resultado_b
    },
    "consulta_c": {
        "pipeline": pipeline_c,
        "resultado": resultado_c
    }
}
```

Guarden `entrega_tc1/02_atlas_consultas.json`.

**No incluyan URI, usuario ni contraseña en ese JSON.**


In [ ]:
# ============================================================
# ETAPA 2.6 — SU CÓDIGO
# ============================================================

atlas_resultados = {}

# Construyan el diccionario y guárdenlo como 02_atlas_consultas.json.


---
# ETAPA 3 — Construyan una bandeja histórica NUEVA desde Atlas
**10 puntos**

## Qué cambia respecto a clase

No van a repetir la regla `1.000 → 163 → 77`.

Ahora la pregunta es:

> ¿Cuáles son los procesos recientes, adjudicados, de mayor exposición económica y con baja participación dentro del histórico de 6.000?

Construyan `pipeline_bandeja` directamente en MongoDB con:

### `$match`

- `anio >= 2024`
- `adjudicado == true`
- `precio_base >= 50000000`
- `respuestas <= 1`

### `$project`

Conserven y oculten `_id`:

- `id_proceso`
- `entidad`
- `nit_entidad`
- `departamento`
- `fecha_publicacion`
- `anio`
- `precio_base`
- `respuestas`
- `proveedor`
- `nit_proveedor`
- `url`

### `$sort`

1. `precio_base DESC`
2. `fecha_publicacion DESC`
3. `id_proceso ASC`

### `$limit`

100

Después:

```python
bandeja_documentos = list(coleccion.aggregate(pipeline_bandeja))
bandeja_historica = pd.DataFrame(bandeja_documentos)
```

Guarden `entrega_tc1/03_bandeja_historica.csv`.

> La bandeja prioriza revisión. No demuestra fraude ni irregularidad.


In [ ]:
# ============================================================
# ETAPA 3 — SU CÓDIGO
# ============================================================

pipeline_bandeja = []
bandeja_documentos = []
bandeja_historica = None

# Construyan el pipeline, ejecútenlo sobre Atlas y exporten el CSV.


---
# ETAPA 4 — Diseñen Cassandra para una consulta que NO vieron en clase
**15 puntos**

## Necesidad operacional

> Para un `anio` y un `departamento`, mostrar hasta 10 procesos de la bandeja empezando por los más recientes. Si dos procesos tienen la misma fecha, mostrar primero el de mayor `precio_base`; si persiste el empate, ordenar por `id_proceso` ascendente.

## Paso 4.1 — Preparar los datos

Creen `bandeja_cassandra` desde `bandeja_historica` con:

- `anio`
- `departamento`
- `fecha_publicacion`
- `precio_base`
- `id_proceso`
- `entidad`
- `respuestas`
- `proveedor`
- `url`

Conviertan `fecha_publicacion` a `datetime`.

## Paso 4.2 — Escribir CQL

La tabla debe llamarse:

```text
tc1.procesos_por_anio_departamento
```

Creen `cql_create`.

El modelo debe responder la consulta **sin `ALLOW FILTERING`**.

No copien la clave de la sesión 5: aquí la partición y el orden responden a otra pregunta.

## Paso 4.3 — Probar el diseño sin depender de Astra

Implementen:

```python
def consulta_cassandra_simulada(df, anio, departamento, n=10):
    ...
```

Debe:

1. filtrar por `anio + departamento`;
2. ordenar `fecha_publicacion DESC`, `precio_base DESC`, `id_proceso ASC`;
3. devolver `n` filas.

Por código encuentren la partición `(anio, departamento)` con más filas y guárdenla como:

```python
particion_prueba = (anio, departamento)
```

Ejecuten la función con `n=10` y guarden `top10_cassandra`.

Exporten `04_modelo_cassandra.cql`.


In [ ]:
# ============================================================
# ETAPA 4 — SU CÓDIGO
# ============================================================

bandeja_cassandra = None

cql_create = """
-- Escriban aquí el CREATE TABLE
"""

def consulta_cassandra_simulada(df, anio, departamento, n=10):
    return None

particion_prueba = None
top10_cassandra = None

# Construyan, prueben y guarden el CQL.


---
# ETAPA 5 — Construyan contexto relacional tipo Neo4j desde SU histórico
**20 puntos**

## Pregunta

> ¿Qué proveedores conectan a la entidad con mayor número de procesos adjudicados con otras entidades dentro de estos 6.000 registros?

No usen el NIT ancla de la sesión 6. Deben derivar uno nuevo.

## Paso 5.1 — Crear `hist_adjudicado`

Desde `historico` conserven filas donde:

- `adjudicado == True`;
- `nit_proveedor` no sea nulo;
- `nit_proveedor` no sea vacío;
- `nit_proveedor` no sea `"No Definido"`.

## Paso 5.2 — Elegir el ancla

Agrupen por `nit_entidad` y cuenten `id_proceso` distintos.

Ordenen:

1. número de procesos DESC;
2. `nit_entidad` ASC.

La primera fila define:

- `nit_ancla`
- `entidad_ancla`

No escriban esos valores manualmente.

## Paso 5.3 — Medir proveedores compartidos

Para los proveedores del ancla calculen:

- `procesos_con_ancla`: procesos distintos con el ancla;
- `entidades_conectadas`: entidades distintas conectadas al mismo proveedor en todo el histórico.

Construyan `resultado_relacional` y ordenen:

1. `entidades_conectadas DESC`
2. `procesos_con_ancla DESC`
3. `nit_proveedor ASC`

Guarden `05_resultado_relacional.csv`.

## Paso 5.4 — Escribir Cypher

Creen estas cuatro cadenas:

- `cypher_carga`: debe usar `UNWIND $rows` y `MERGE` de `Entidad`, `Proceso`, `Proveedor`, más `PUBLICA` y `ADJUDICADO_A`;
- `cypher_contexto`: procesos y proveedores para `$nit_ancla`;
- `cypher_compartidos`: otras entidades que comparten proveedor con el ancla;
- `cypher_ranking`: proveedores por `count(DISTINCT ...)`.

Guarden las cuatro consultas en `05_neo4j_consultas.cypher`.

## Paso 5.5 — Verificar el subgrafo con NetworkX

Construyan `G = nx.DiGraph()` para el ancla.

IDs:

- `E:<nit_entidad>`
- `P:<id_proceso>`
- `V:<nit_proveedor>`

Aristas:

- Entidad → Proceso, `relacion="PUBLICA"`
- Proceso → Proveedor, `relacion="ADJUDICADO_A"`

Guarden:

```python
nodos_grafo = G.number_of_nodes()
aristas_grafo = G.number_of_edges()
```


In [ ]:
# ============================================================
# ETAPA 5 — SU CÓDIGO
# ============================================================

import networkx as nx

hist_adjudicado = None
ranking_entidades = None
nit_ancla = None
entidad_ancla = None
prov_ancla = None
prov_global = None
resultado_relacional = None

cypher_carga = """
// carga
"""
cypher_contexto = """
// contexto
"""
cypher_compartidos = """
// compartidos
"""
cypher_ranking = """
// ranking
"""

G = nx.DiGraph()
nodos_grafo = 0
aristas_grafo = 0

# Resuelvan los cinco pasos y guarden los dos archivos de la etapa.


---
# ETAPA 6 — Expliquen qué construyeron
**5 puntos**

Creen `informe_tecnico` y guárdenlo como `entrega_tc1/06_informe_tecnico.md`.

Debe tener exactamente estas secciones:

```text
## 1. Cómo construimos el histórico
## 2. Qué comprobamos en MongoDB Atlas
## 3. Cómo funciona la bandeja histórica
## 4. Por qué el modelo Cassandra responde la consulta
## 5. Qué relaciones aporta Neo4j y qué no podemos concluir
```

El texto debe incorporar mediante variables:

- número de archivos integrados;
- documentos cargados en Atlas;
- resultado de la consulta A;
- tamaño de `bandeja_historica`;
- `nit_ancla`;
- máximo de `entidades_conectadas`.

Incluyan explícitamente que la bandeja y las conexiones contractuales **no demuestran por sí solas fraude, favorecimiento ni colusión**.


In [ ]:
# ============================================================
# ETAPA 6 — SU INFORME
# ============================================================

max_entidades_conectadas = (
    int(resultado_relacional["entidades_conectadas"].max())
    if isinstance(resultado_relacional, pd.DataFrame) and not resultado_relacional.empty
    else None
)

informe_tecnico = f"""
## 1. Cómo construimos el histórico
[Completen usando sus variables]

## 2. Qué comprobamos en MongoDB Atlas
[Completen usando sus variables]

## 3. Cómo funciona la bandeja histórica
[Completen usando sus variables]

## 4. Por qué el modelo Cassandra responde la consulta
[Completen]

## 5. Qué relaciones aporta Neo4j y qué no podemos concluir
[Completen]
"""

# Guarden 06_informe_tecnico.md.


---
# VALIDACIÓN Y ENTREGA

Ejecuten esta celda solo cuando las seis etapas estén terminadas.

El validador revisa:

- que realmente integraron seis fragmentos;
- que el JSON tenga el esquema solicitado;
- que la colección usada sea una colección de PyMongo y contenga los documentos;
- que las consultas A, B y C produzcan el mismo resultado que una referencia;
- que la bandeja sea el resultado del pipeline solicitado;
- que Cassandra responda el patrón nuevo;
- que el ancla Neo4j se derive del histórico;
- que el paquete de archivos esté completo.

Al final genera:

- `manifest_tc1.json`
- puntaje / 100
- nota / 5.0
- SHA-256
- `TC1_<pareja>.zip`


In [ ]:
# ============================================================
# VALIDADOR VERSIONADO — NO EDITAR
# ============================================================

VALIDATOR_COMMIT = "ad0cfbce7a81d70dfa06856c4f7ad3b03267c34a"
VALIDATOR_URL = (
    "https://raw.githubusercontent.com/jazaineam1/BigData2026/"
    f"{VALIDATOR_COMMIT}/utils/tc1_validator.py"
)

codigo_validador = urllib.request.urlopen(VALIDATOR_URL).read().decode("utf-8")
espacio_validador = {}
exec(codigo_validador, espacio_validador)

manifest_tc1 = espacio_validador["evaluar"](globals())


## Checklist antes de entregar

- [ ] cargamos seis CSV por código;
- [ ] construimos `historico` y `documentos`;
- [ ] generamos el JSON de 6.000 registros;
- [ ] hicimos `ping` contra Atlas;
- [ ] cargamos nuestra colección real;
- [ ] ejecutamos Consulta A, B y C;
- [ ] generamos la bandeja desde Atlas;
- [ ] diseñamos Cassandra para la consulta nueva;
- [ ] derivamos el ancla Neo4j desde nuestros datos;
- [ ] generamos los archivos de entrega;
- [ ] ejecutamos el validador;
- [ ] descargamos el ZIP y el `.ipynb` ejecutado.

**Git/GitHub es opcional.**
